# ============================================================
# STEP 5 — FEATURE-GROUP ABLATION (TUMC)
# ============================================================
# Quantifies how much each FEATURE GROUP contributes, including
# the new turkish-extended group. Two complementary views:
#
#   INCREMENTAL (cumulative): start with lexical, add one group
#     at a time, report the marginal gain of each addition.
#       A1 lexical -> A2 +adversarial -> A3 +turkish-base
#       -> A4 +turkish-extended -> A5 +graph
#
#   LEAVE-ONE-GROUP-OUT (LOGO): full set minus each group, report
#     the drop = that group's UNIQUE contribution.
#
# Deployment config: inductive graph, Experiment C (artifacts +
# 2 leaky graph features removed). Both tasks, HistGB. Keyword
# group tracked SEPARATELY (it is partly circular) and excluded
# from the "honest" feature groups by default; a toggle includes
# it so the circular contribution is visible.
#
# Output: step5_incremental.csv, step5_logo.csv,
#         step5_ablation_table.tex, step5_ablation.png
# ============================================================

In [ ]:
import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, roc_auc_score, matthews_corrcoef

warnings.filterwarnings("ignore")
SEED = 42
DATA_DIR = "/content/drive/MyDrive/TUMC_dataset"
for cand in ["features_full_final_inductive_dedup.csv","features_full_final_inductive.csv","features_full_v12.csv"]:
    INPUT = os.path.join(DATA_DIR, cand)
    if os.path.exists(INPUT): break

INCLUDE_KEYWORDS = False   # keep circular keyword group out of honest groups
LEAKY = ["cluster_malicious_ratio","campaign_token_score","is_tr_domain","is_https"]

# ── Define feature groups by name patterns ───────────────────
# Adjust these lists to match actual column names if needed.
TURKISH_EXT = ["tr_phishing_vocab_score","tr_transliteration_score",
               "tr_brand_impersonation_score","tr_semantic_urgency_score",
               "tr_scam_vocab_score"]
KEYWORD = ["has_phishing_keyword","num_phishing_keywords","has_malware_keyword",
           "num_malware_keywords","has_scam_keyword","num_scam_keywords",
           "has_turkish_keyword","num_turkish_keywords"]
GRAPH_HINTS = ["token_cluster","ngram_cluster","centrality","shared_token",
               "domain_family","sibling_domain","tld_token_cooccur","template_reuse",
               "rare_token","token_reuse","unique_token_ratio","hub_domain",
               "host_pattern","subdomain_reuse","campaign_member","domain_ngram"]
TURKISH_BASE_HINTS = ["tr_char","tr_stopword","tr_suffix","vowel_harmony","tr_token",
                      "tr_bigram","tr_vs_en","langid_tr","is_turkish_dominant",
                      "has_tr_bank","has_tr_gov","has_tr_ecommerce","has_tr_telecom",
                      "tr_sector","distinct_tr","has_tr_char"]
ADVERSARIAL_HINTS = ["brand","edit_dist","homoglyph","punycode","typosquat","squat",
                     "lookalike","confusable","suspicious_tld","is_free_hosting",
                     "registrant","obfusc","redirect","shorten"]

def classify_group(col):
    c = col.lower()
    if col in TURKISH_EXT: return "turkish_ext"
    if col in KEYWORD: return "keyword"
    if any(h in c for h in GRAPH_HINTS): return "graph"
    if any(h in c for h in TURKISH_BASE_HINTS): return "turkish_base"
    if any(h in c for h in ADVERSARIAL_HINTS): return "adversarial"
    return "lexical"   # default bucket

print("="*70)
print("STEP 5 — FEATURE-GROUP ABLATION (HistGB, inductive, Exp C)")
print("="*70)

df = pd.read_csv(INPUT, low_memory=False)
META = {"url","source","tld","label","label_enc","class_final","class_enc",
        "fold","reg_domain"}
ALL_FEATS = [c for c in df.columns if c not in META and c not in LEAKY]

groups = {}
for col in ALL_FEATS:
    g = classify_group(col)
    if g == "keyword" and not INCLUDE_KEYWORDS: continue
    groups.setdefault(g, []).append(col)

print("\n  Feature groups:")
for g in ["lexical","adversarial","turkish_base","turkish_ext","graph","keyword"]:
    if g in groups: print(f"    {g:<14s}: {len(groups[g])} features")

folds = df["fold"].values

def evaluate(feature_list, task):
    """5-fold inductive OOF score for a feature subset."""
    if not feature_list: return np.nan
    X = df[feature_list].fillna(0).values
    y = df["class_enc"].values if task=="5class" else df["label_enc"].values
    oof = np.zeros(len(df)); oof_proba = None
    preds = np.zeros(len(df), dtype=int)
    if task=="binary": proba = np.zeros(len(df))
    for fid in sorted(set(folds)):
        tr = np.where(folds!=fid)[0]; te = np.where(folds==fid)[0]
        m = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.05,
            max_iter=300, random_state=SEED)
        m.fit(X[tr], y[tr])
        preds[te] = m.predict(X[te])
        if task=="binary":
            proba[te] = m.predict_proba(X[te])[:,1]
    if task=="5class":
        return f1_score(y, preds, average="macro", zero_division=0)
    else:
        return roc_auc_score(y, proba)

# ── INCREMENTAL ──────────────────────────────────────────────
ORDER = ["lexical","adversarial","turkish_base","turkish_ext","graph"]
ORDER = [g for g in ORDER if g in groups]
print(f"\n[INCREMENTAL] cumulative add: {' -> '.join(ORDER)}")
inc_rows = []
for task in ["binary","5class"]:
    cum = []
    prev = None
    print(f"\n  Task: {task}")
    for g in ORDER:
        cum = cum + groups[g]
        score = evaluate(cum, task)
        gain = (score - prev) if prev is not None else score
        metric = "AUC" if task=="binary" else "macroF1"
        print(f"    +{g:<14s} ({len(cum):>3d} feats): {metric}={score:.4f}  "
              f"marginal={gain:+.4f}")
        inc_rows.append({"task":task,"step":f"+{g}","n_features":len(cum),
                         "score":round(score,4),"marginal_gain":round(gain,4)})
        prev = score
pd.DataFrame(inc_rows).to_csv(os.path.join(DATA_DIR,"step5_incremental.csv"), index=False)

# ── LEAVE-ONE-GROUP-OUT ──────────────────────────────────────
HONEST = [g for g in ORDER]   # all honest groups
full_feats = [f for g in HONEST for f in groups[g]]
print(f"\n[LEAVE-ONE-GROUP-OUT] full ({len(full_feats)} feats) minus each group")
logo_rows = []
for task in ["binary","5class"]:
    print(f"\n  Task: {task}")
    full_score = evaluate(full_feats, task)
    metric = "AUC" if task=="binary" else "macroF1"
    print(f"    FULL ({len(full_feats)} feats): {metric}={full_score:.4f}")
    for g in HONEST:
        keep = [f for gg in HONEST if gg!=g for f in groups[gg]]
        score = evaluate(keep, task)
        drop = full_score - score
        print(f"    full - {g:<14s} ({len(keep):>3d} feats): {metric}={score:.4f}  "
              f"drop={drop:+.4f}")
        logo_rows.append({"task":task,"removed":g,"n_features":len(keep),
                          "score":round(score,4),"drop_vs_full":round(drop,4)})
    logo_rows.append({"task":task,"removed":"(none/full)","n_features":len(full_feats),
                      "score":round(full_score,4),"drop_vs_full":0.0})
pd.DataFrame(logo_rows).to_csv(os.path.join(DATA_DIR,"step5_logo.csv"), index=False)

# ── Plot (incremental marginal gains, 5-class) ───────────────
inc = pd.DataFrame(inc_rows); inc5 = inc[inc["task"]=="5class"]
fig, ax = plt.subplots(figsize=(9,5))
ax.bar(range(len(inc5)), inc5["score"], color="#1D9E75", alpha=0.85)
ax.set_xticks(range(len(inc5))); ax.set_xticklabels(inc5["step"], rotation=30, ha="right")
ax.set_ylabel("Macro-$F_1$ (5-class)"); ax.set_ylim(0.5,1.0)
for i,(s,g) in enumerate(zip(inc5["score"],inc5["marginal_gain"])):
    ax.text(i, s+0.005, f"+{g:.3f}", ha="center", fontsize=8)
ax.set_title("Incremental Feature-Group Ablation (HistGB, 5-class, inductive)",
             fontweight="bold")
plt.tight_layout(); plt.savefig(os.path.join(DATA_DIR,"step5_ablation.png"), dpi=150)
plt.close()

# ── LaTeX table (incremental) ────────────────────────────────
tex = ["\\begin{table}[htbp]","\\centering",
 "\\caption{Incremental feature-group ablation (HistGB, inductive graph, Experiment~C). Each row adds one feature group cumulatively; the marginal column reports the change from the previous row. The graph group provides the dominant gain, while the Turkish-extended group is marginal---evidence that campaign-infrastructure features, not vocabulary, drive detection.}",
 "\\label{tab:group-ablation}","\\footnotesize","\\begin{tabular}{lrcc}","\\toprule",
 "Cumulative group & \\#feat & Binary AUC & 5-class macro-$F_1$ \\\\","\\midrule"]
incb = {r["step"]:r for r in inc_rows if r["task"]=="binary"}
inc5d = {r["step"]:r for r in inc_rows if r["task"]=="5class"}
for g in ORDER:
    step=f"+{g}"
    b=incb.get(step,{}); f5=inc5d.get(step,{})
    tex.append(f"{step.replace('_','-')} & {f5.get('n_features','')} & "
               f"{b.get('score','')} & {f5.get('score','')} \\\\")
tex += ["\\bottomrule","\\end{tabular}","\\end{table}"]
open(os.path.join(DATA_DIR,"step5_ablation_table.tex"),"w").write("\n".join(tex))

print(f"\n{'='*70}\nSTEP 5 COMPLETE\n{'='*70}")
print("  Saved: step5_incremental.csv, step5_logo.csv,")
print("         step5_ablation_table.tex, step5_ablation.png")
print("  Honest expectation: large graph gain, ~0 turkish_ext gain.")
print("  A flat turkish_ext line is EVIDENCE for the graph-driven thesis.")